# 10.19 — Video & 3D generation
Video and 3D generation add consistency constraints across time, viewpoints, and geometry. Diffusion and flow-like engines provide samples, but motion and projection keep frames coherent. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

SEED = 1014
rng = np.random.default_rng(SEED)


def normalize01(x):
    arr = np.asarray(x, dtype=float)
    lo = float(np.min(arr))
    hi = float(np.max(arr))
    span = hi - lo
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / span


def synthetic_faces(n=64, size=16, seed=SEED):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    faces = []
    for i in range(n):
        face = 0.25 + 0.25 * np.exp(-2.0 * (xx ** 2 + yy ** 2))
        eye_y = -0.30 + local_rng.normal(0.0, 0.03)
        eye_dx = 0.35 + local_rng.normal(0.0, 0.02)
        mouth_y = 0.45 + local_rng.normal(0.0, 0.04)
        face -= 0.35 * np.exp(-90.0 * ((xx - eye_dx) ** 2 + (yy - eye_y) ** 2))
        face -= 0.35 * np.exp(-90.0 * ((xx + eye_dx) ** 2 + (yy - eye_y) ** 2))
        face += 0.14 * np.exp(-70.0 * (xx ** 2 + (yy - 0.06) ** 2))
        face -= 0.20 * np.exp(-55.0 * (xx ** 2 + (yy - mouth_y) ** 2))
        face += local_rng.normal(0.0, 0.025, size=(size, size))
        faces.append(normalize01(face))
    return np.asarray(faces).reshape(n, size * size)


def load_tiny_faces():
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(download_if_missing=False, shuffle=True, random_state=SEED)
        data = faces.data[:64]
        return data, (64, 64), "Olivetti faces cache"
    except Exception:
        return synthetic_faces(), (16, 16), "synthetic face fallback"


def make_f9_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    d1 = local_rng.normal(0.0, 1.0, size=(80, 1))
    d2, _ = make_moons(n_samples=120, noise=0.08, random_state=seed)
    d3, _ = make_blobs(n_samples=144, centers=4, n_features=6, cluster_std=0.65, random_state=seed)
    digits = load_digits()
    d4 = digits.data[:80] / 16.0
    faces, face_shape, face_source = load_tiny_faces()
    ladder = [
        {"name": "D1 gaussian", "data": d1, "shape": (1, 1), "kind": "vector"},
        {"name": "D2 two moons", "data": d2, "shape": (1, 2), "kind": "points"},
        {"name": "D3 mixture", "data": d3, "shape": (2, 3), "kind": "vector"},
        {"name": "D4 sklearn digits", "data": d4, "shape": (8, 8), "kind": "image"},
        {"name": "D5 faces (" + face_source + ")", "data": faces, "shape": face_shape, "kind": "image"},
    ]
    return ladder


def centered_scaled(data):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(np.asarray(data, dtype=float))
    return scaled, scaler


def choose_components(data, cap=8):
    n_samples, n_features = data.shape
    return max(1, min(cap, n_samples - 1, n_features))


def pca_reconstruct(data, n_components):
    n_components = max(1, min(n_components, data.shape[0] - 1, data.shape[1]))
    pca = PCA(n_components=n_components, random_state=SEED)
    z = pca.fit_transform(data)
    recon = pca.inverse_transform(z)
    return recon, z, pca


def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def two_sample_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b), 80)
    a = a[:n]
    b = b[:n]
    aa = pairwise_distances(a, a).mean()
    bb = pairwise_distances(b, b).mean()
    ab = pairwise_distances(a, b).mean()
    return float(2.0 * ab - aa - bb)


def preview_array(ax, sample, shape, title):
    arr = np.asarray(sample)
    if int(np.prod(shape)) == arr.size and shape[0] > 1 and shape[1] > 1:
        ax.imshow(arr.reshape(shape), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.plot(arr.ravel(), marker="o")
    ax.set_title(title, fontsize=9)


def show_ladder_preview(ladder):
    fig, axes = plt.subplots(1, len(ladder), figsize=(14, 2.8))
    for ax, rung in zip(axes, ladder):
        preview_array(ax, rung["data"][0], rung["shape"], rung["name"])
    plt.tight_layout()
    return fig


## The concept, built once (D1)
A toy 3D motion model is $p_t=p_0+t v$ before projection into frames. With $p_0=(0,0,0)$, $v=(1,0.5,0.2)$, and $t=3$, the lesson gives $p_3=(3.000,1.500,0.6000)$ and speed $\lVert vVert_2=1.136$.

In [ ]:

def temporal_geometric_generator(p0, velocity, t):
    p0 = np.asarray(p0, dtype=float)
    velocity = np.asarray(velocity, dtype=float)
    pt = p0 + t * velocity
    speed = float(np.linalg.norm(velocity))
    return pt, speed

p3, speed = temporal_geometric_generator([0.0, 0.0, 0.0], [1.0, 0.5, 0.2], 3.0)
assert np.allclose(p3, np.array([3.0, 1.5, 0.6]))
assert round(speed, 3) == 1.136
print("p3", np.round(p3, 4), "speed", round(speed, 3))


For each rung, the generator creates a tiny frame sequence by shifting a sample along a shared trajectory. The metric combines reconstruction and temporal consistency, so sharp independent frames do not win if they flicker.

In [ ]:

def make_frame_sequence(sample, shape, steps=5, velocity=(1, 0)):
    arr = np.asarray(sample, dtype=float).ravel()
    if int(np.prod(shape)) != arr.size or shape[0] <= 1 or shape[1] <= 1:
        base = np.resize(arr, (1, arr.size))
        frames = []
        for t in range(steps):
            frames.append(np.roll(base, shift=t, axis=1).ravel())
        return np.asarray(frames)
    image = arr.reshape(shape)
    frames = []
    for t in range(steps):
        shifted = np.roll(image, shift=int(t * velocity[0]), axis=1)
        shifted = np.roll(shifted, shift=int(t * velocity[1]), axis=0)
        frames.append(shifted.ravel())
    return np.asarray(frames)


def video_consistency_generate(data, shape, seed=SEED, jitter=0.05):
    local_rng = np.random.default_rng(seed)
    base = normalize01(data[0])
    clean = make_frame_sequence(base, shape, steps=5, velocity=(1, 0))
    generated = clean + local_rng.normal(0.0, jitter, size=clean.shape)
    reconstruction = mse(clean, generated)
    temporal_clean = np.diff(clean, axis=0)
    temporal_generated = np.diff(generated, axis=0)
    consistency = mse(temporal_clean, temporal_generated)
    metric = reconstruction + consistency
    return generated, metric, reconstruction, consistency


## The dataset ladder (D1–D5)
We use the same F9 ladder inline for every topic: a one-dimensional Gaussian, two moons, a mixture, tiny sklearn digits, and a face rung. The face rung uses a local Olivetti cache when present and otherwise a deterministic no-download synthetic face fallback.

In [ ]:

ladder = make_f9_ladder()
for index, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(f"D{index}: {rung['name']} | shape={data.shape} | sample_shape={rung['shape']} | kind={rung['kind']}")
show_ladder_preview(ladder)


## Run the SAME method across D1-D5
The same reusable method is applied to every rung and one plan metric is collected per rung.

In [ ]:

results = []
for level, rung in enumerate(ladder, start=1):
    frames, metric, reconstruction, consistency = video_consistency_generate(rung["data"], rung["shape"], seed=SEED + level, jitter=0.04)
    results.append({"level": level, "name": rung["name"], "metric": metric, "sample": frames[-1], "shape": rung["shape"]})
    print(f"D{level} {rung['name']}: reconstruction/consistency error={metric:.5f}, temporal={consistency:.5f}")


## Results visualization
The closing figure has generated-sample panels for D1-D5 plus the requested metric curve.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(14, 2.8))
for ax, result in zip(axes, results):
    preview_array(ax, result["sample"], result["shape"], result["name"])
plt.suptitle("Generated or reconstructed samples by rung")
plt.tight_layout()

plt.figure(figsize=(6, 3.2))
plt.plot([r["level"] for r in results], [r["metric"] for r in results], marker="o")
plt.xticks([r["level"] for r in results], [r["name"].split()[0] for r in results])
plt.ylabel("reconstruction / consistency error")
plt.xlabel("F9 rung")
plt.title("Temporal geometric generator: metric vs. complexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## Pitfall on D5: frame quality alone or ignored geometry
A generator can make every frame look sharp while the latent trajectory jitters independently. We compare per-frame noisy D5 panels with a shared temporal/geometric path.

In [ ]:

d5 = ladder[-1]
local_rng = np.random.default_rng(SEED)
clean = make_frame_sequence(normalize01(d5["data"][0]), d5["shape"], steps=5, velocity=(1, 0))
flicker = clean + local_rng.normal(0.0, 0.16, size=clean.shape)
steady = clean + local_rng.normal(0.0, 0.035, size=clean.shape)
flicker_error = mse(clean, flicker) + mse(np.diff(clean, axis=0), np.diff(flicker, axis=0))
steady_error = mse(clean, steady) + mse(np.diff(clean, axis=0), np.diff(steady, axis=0))
print(f"frame-only flicker error={flicker_error:.5f}")
print(f"shared-trajectory error={steady_error:.5f}")
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i in range(5):
    preview_array(axes[0, i], flicker[i], d5["shape"], f"flicker {i}")
    preview_array(axes[1, i], steady[i], d5["shape"], f"steady {i}")
plt.tight_layout()


## Evaluate it + Practice
- Track `reconstruction/consistency error` against a no-skill baseline such as random samples, unconditioned reconstruction, or independent frames.
- Run a cheap sanity check: dimensions match, finite metrics, and generated samples stay in the training range.
- Ablation: score frames independently and remove the shared motion trajectory; the metric should get worse.
- Failure signals: D5 looks plausible in one panel but the metric curve jumps, indicating artifacts hidden by small previews.

Practice 1: change the latent or conditioning dimension and predict which rung changes most.


Practice 2: change the velocity vector and verify the projected frame shifts.

Practice 3: add a second viewpoint by using vertical and horizontal shifts together.